In [1]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset
from torchvision.io import read_image

class MassHumanBurns(Dataset):
    _views_per_burn = 4
    _pairs_per_burn = _views_per_burn ** 2
    def __init__(self, root_dir, return_images=True, return_full_masks=True, return_burn_masks=True):
        self.root_dir = root_dir
        self.labels = torch.tensor(np.load(os.path.join(root_dir, "labels.npy")), dtype=torch.float32)
        self.return_images = return_images
        self.return_full_masks = return_full_masks
        self.return_burn_masks = return_burn_masks
        self._len = self.labels.numel() * self._pairs_per_burn
        self._burn_levels = self.labels.shape[1]
        self._pairs_per_human = self._burn_levels * self._pairs_per_burn

    def __len__(self):
        return self._len

    def __getitem__(self, idx):
        human_idx = idx // self._pairs_per_human
        burn_idx = idx % self._pairs_per_human // self._pairs_per_burn
        front_view_idx = burn_idx % self._pairs_per_human % self._pairs_per_burn // self._views_per_burn
        back_view_idx = burn_idx % self._pairs_per_human % self._pairs_per_burn % self._views_per_burn
        res = {"label": self.labels[human_idx, burn_idx]}
        if self.return_images:
            res["images"] = (
                read_image(os.path.join(self.root_dir, "images", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.jpg")), 
                read_image(os.path.join(self.root_dir, "images", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.jpg"))
            )
        if self.return_full_masks:
            res["full_masks"] = (
                read_image(os.path.join(self.root_dir, "full_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "full_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        if self.return_burn_masks:
            res["burn_masks"] = (
                read_image(os.path.join(self.root_dir, "burn_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "burn_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        return res

In [2]:
from torch.utils.data import Subset

full_set = MassHumanBurns("mass_human_burns", return_images=False)
train_indices = torch.tensor([list(range(i*625*full_set._pairs_per_human,(i*625+500)*full_set._pairs_per_human)) for i in range(8)]).flatten()
test_indices = torch.tensor([list(range((i*625+500)*full_set._pairs_per_human,(i*625+625)*full_set._pairs_per_human)) for i in range(8)]).flatten()

test_set = Subset(full_set, test_indices)


In [3]:
class BaselineTest(Dataset):
    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]
        full = sample["full_masks"][0].to(torch.bool).sum().item()+sample["full_masks"][1].to(torch.bool).sum().item()
        burn = sample["burn_masks"][0].to(torch.bool).sum().item()+sample["burn_masks"][1].to(torch.bool).sum().item()
        label = sample["label"].item()
        return burn/full, label, idx

In [4]:
from torch.utils.data import DataLoader

dataloader = DataLoader(BaselineTest(test_set), num_workers=8, in_order=False)

In [5]:
from tqdm import tqdm

pre_list = torch.empty([1000,8,16])
label_list = torch.empty([1000,8,16])
for b in tqdm(dataloader):
    idx = b[2].item()
    human_idx = idx // full_set._pairs_per_human
    burn_idx = idx % full_set._pairs_per_human // full_set._pairs_per_burn
    pair_idx = idx % full_set._pairs_per_human % full_set._pairs_per_burn
    pre_list[human_idx,burn_idx,pair_idx] = b[0].item()
    label_list[human_idx,burn_idx,pair_idx] = b[1].item()

torch.save(pre_list, "baseline_pre.pt")

100%|██████████| 128000/128000 [02:34<00:00, 830.12it/s]


In [6]:
from sklearn.metrics import r2_score
print("R2 Score:", r2_score(label_list.flatten(), pre_list.flatten()))

R2 Score: 0.9919223785400391


In [7]:
triclass_label = torch.zeros_like(label_list)
triclass_label[label_list < 0.05] = 0
triclass_label[(label_list >= 0.05) & (label_list < 0.2)] = 1
triclass_label[label_list >= 0.2] = 2
triclass_pre = torch.zeros_like(pre_list)
triclass_pre[pre_list < 0.05] = 0
triclass_pre[(pre_list >= 0.05) & (pre_list < 0.2)] = 1
triclass_pre[pre_list >= 0.2] = 2
from sklearn.metrics import classification_report
print(classification_report(triclass_label.flatten(), triclass_pre.flatten(), digits=3))

              precision    recall  f1-score   support

         0.0      0.946     0.912     0.928     15600
         1.0      0.939     0.926     0.933     32640
         2.0      0.980     0.993     0.987     79760

    accuracy                          0.966    128000
   macro avg      0.955     0.944     0.949    128000
weighted avg      0.966     0.966     0.966    128000



In [8]:
mae = (pre_list-label_list).abs()
print(f"MAE:{mae.mean().item()*100:.2f}, STD:{torch.std(mae).item()*100:.2f}")

MAE:1.71, STD:1.46


In [9]:
mre = (pre_list-label_list).abs()/(label_list)
print(f"MRE:{mre.mean().item()*100:.2f}, STD:{torch.std(mre).item()*100:.2f}")

MRE:8.33, STD:9.16


In [10]:
import pandas as pd
import pingouin as pg

reshaped_pre = pre_list.view(-1, full_set._pairs_per_burn)
n_subjects, n_raters = reshaped_pre.shape

long_data_list = []
for i in range(n_raters):
    rater_name = f'Observer_{i+1}'
    temp_df = pd.DataFrame({
        'subject': torch.arange(n_subjects),
        'rater': rater_name,
        'rating': reshaped_pre[:, i]
    })
    long_data_list.append(temp_df)

long_data = pd.concat(long_data_list, ignore_index=True)
pg.intraclass_corr(
    data=long_data,
    targets='subject',
    raters='rater',
    ratings='rating'
)

/data/BurnAreaNet/.env/lib/python3.12/site-packages/pingouin/parametric.py:1014: RuntimeWarning: divide by zero encountered in scalar divide
  fval = msbetween / mserror


,Type,Description,ICC,F,df1,df2,pval,CI95
0,ICC1,Single raters absolute,1.0,-2.697000e+08,7999,120000,1.0,"[1.0, 1.0]"
1,ICC2,Single random raters,1.0,-2.696663e+08,7999,119985,1.0,"[1.0, 1.0]"
2,ICC3,Single fixed raters,1.0,-2.696663e+08,7999,119985,1.0,"[1.0, 1.0]"
3,ICC1k,Average raters absolute,1.0,-2.697000e+08,7999,120000,1.0,"[1.0, 1.0]"
4,ICC2k,Average random raters,1.0,-2.696663e+08,7999,119985,1.0,"[1.0, 1.0]"
5,ICC3k,Average fixed raters,1.0,-2.696663e+08,7999,119985,1.0,"[1.0, 1.0]"


In [11]:
for i in range(8):
    print(f"Burn Level {i+1}:", end=" ")
    print(f"MAE: {mae[:, i].mean().item()*100:.2f}±{torch.std(mae[:, i]).item()*100:.2f}", end=", ")
    print(f"MRE: {mre[:, i].mean().item()*100:.2f}±{torch.std(mre[:, i]).item()*100:.2f}")

Burn Level 1: MAE: 0.45±0.42, MRE: 19.76±16.11
Burn Level 2: MAE: 0.92±0.69, MRE: 12.27±9.25
Burn Level 3: MAE: 1.33±1.00, MRE: 9.13±6.90
Burn Level 4: MAE: 1.73±1.25, MRE: 7.08±5.13
Burn Level 5: MAE: 2.01±1.50, MRE: 5.86±4.33
Burn Level 6: MAE: 2.19±1.61, MRE: 4.97±3.66
Burn Level 7: MAE: 2.51±1.70, MRE: 4.34±2.96
Burn Level 8: MAE: 2.56±1.49, MRE: 3.26±1.99


In [12]:
import json

with open("mass_human_burns/gender.json", "r") as f:
    gender_data = json.load(f)

gender_mask = torch.zeros(5000, dtype=bool)
gender_mask[torch.tensor(gender_data["male"])]=1

test_human_indices = (test_indices // test_set.dataset._pairs_per_human).view([1000,8,16])[:,0,0]
test_gender_mask = gender_mask[test_human_indices]

print(f"Number of males: {test_gender_mask.sum().item()}, Number of females: {(~test_gender_mask).sum().item()}")

print(f"Male:", end=" ")
print(f"MAE: {mae[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(mae[test_gender_mask==1]).item()*100:.2f}", end=", ")
print(f"MRE: {mre[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(mre[test_gender_mask==1]).item()*100:.2f}", end="")
print(f"Female:", end=" ")
print(f"MAE: {mae[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(mae[test_gender_mask==0]).item()*100:.2f}", end=", ")
print(f"MRE: {mre[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(mre[test_gender_mask==0]).item()*100:.2f}", end="")

print(pg.ttest((mae[test_gender_mask==0].view(-1)*100).numpy().tolist(), (mae[test_gender_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[test_gender_mask==0].view(-1)*100).numpy().tolist(), (mre[test_gender_mask==1].view(-1)*100).numpy().tolist()))

Number of males: 493, Number of females: 507
Male: MAE: 1.76±1.52, MRE: 8.50±9.37Female: MAE: 1.66±1.41, MRE: 8.17±8.95                T            dof alternative         p_val            CI95  \
T_test -12.521974  126683.179269   two-sided  5.945839e-36  [-0.12, -0.09]   

         cohen_d  power       BF10  
T_test  0.070079    1.0  6.665e+31  
               T            dof alternative         p_val            CI95  \
T_test -6.587083  127292.902294   two-sided  4.502901e-11  [-0.44, -0.24]   

        cohen_d     power       BF10  
T_test  0.03685  0.999998  1.657e+07  


In [13]:
age_mask = torch.zeros(1000, dtype=bool)
age_mask[:500] = 1

print(f"Adult:", end=" ")
print(f"MAE: {mae[age_mask==1].mean().item()*100:.2f}±{torch.std(mae[age_mask==1]).item()*100:.2f}", end=", ")
print(f"MRE: {mre[age_mask==1].mean().item()*100:.2f}±{torch.std(mre[age_mask==1]).item()*100:.2f}")
print(f"Child:", end=" ")
print(f"MAE: {mae[age_mask==0].mean().item()*100:.2f}±{torch.std(mae[age_mask==0]).item()*100:.2f}", end=", ")
print(f"MRE: {mre[age_mask==0].mean().item()*100:.2f}±{torch.std(mre[age_mask==0]).item()*100:.2f}")

print(pg.ttest((mae[age_mask==0].view(-1)*100).numpy().tolist(), (mae[age_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[age_mask==0].view(-1)*100).numpy().tolist(), (mre[age_mask==1].view(-1)*100).numpy().tolist()))

Adult: MAE: 1.71±1.49, MRE: 8.35±9.33
Child: MAE: 1.71±1.44, MRE: 8.31±8.99
               T     dof alternative     p_val           CI95   cohen_d  \
T_test -0.531552  127998   two-sided  0.595037  [-0.02, 0.01]  0.002971   

           power   BF10  
T_test  0.082946  0.007  
               T     dof alternative    p_val           CI95   cohen_d  \
T_test -0.787416  127998   two-sided  0.43104  [-0.14, 0.06]  0.004402   

           power   BF10  
T_test  0.123491  0.009  


In [14]:
pose_list = ["A-Pose", "None", "Star", "T-Pose"]
for i in range(4):
    pose_mask = torch.zeros(1000, dtype=bool)
    pose_mask[i*125:(i+1)*125] = 1 # Adult
    pose_mask[i*125+500:(i+1)*125+500] = 1 # Child
    print(f"Pose {pose_list[i]}:", end=" ")
    print(f"MAE: {mae[pose_mask==1].mean().item()*100:.2f}±{torch.std(mae[pose_mask==1]).item()*100:.2f}", end=", ")
    print(f"MRE: {mre[pose_mask==1].mean().item()*100:.2f}±{torch.std(mre[pose_mask==1]).item()*100:.2f}")

Pose A-Pose: MAE: 1.63±1.38, MRE: 8.21±9.46
Pose None: MAE: 1.71±1.48, MRE: 8.46±9.62
Pose Star: MAE: 1.70±1.47, MRE: 8.00±8.47
Pose T-Pose: MAE: 1.81±1.53, MRE: 8.66±9.04
